In [1]:
import pyffx
import os

class DIYTokenizer:
    def __init__(self, secret_key: bytes):
        """
        secret_key: กุญแจลับสำหรับเข้ารหัส (สำคัญมาก ห้ามหลุดเด็ดขาด)
        """
        self.secret_key = secret_key
        
        # ตั้งค่าตัวอักษรที่อนุญาต (Alphabet) ในที่นี้คือตัวเลข 0-9
        self.alphabet = '0123456789'

    def tokenize_thai_id(self, citizen_id: str) -> str:
        """แปลงเลขบัตรประชาชน 13 หลัก เป็น Token 13 หลัก"""
        if len(citizen_id) != 13 or not citizen_id.isdigit():
            raise ValueError("บัตรประชาชนต้องเป็นตัวเลข 13 หลัก")
            
        # กำหนดความยาวให้ FPE Engine
        cipher = pyffx.String(self.secret_key, alphabet=self.alphabet, length=13)
        return cipher.encrypt(citizen_id)

    def detokenize_thai_id(self, token: str) -> str:
        """แปลง Token 13 หลัก คืนเป็นเลขบัตรประชาชนจริง"""
        cipher = pyffx.String(self.secret_key, alphabet=self.alphabet, length=13)
        return cipher.decrypt(token)

    def tokenize_credit_card(self, cc_number: str) -> str:
        """แปลงเลขบัตรเครดิต 16 หลัก เป็น Token 16 หลัก (รักษา 6 หลักแรก และ 4 หลักสุดท้าย)"""
        cc_clean = cc_number.replace("-", "").replace(" ", "")
        if len(cc_clean) != 16:
            raise ValueError("เลขบัตรเครดิตต้องมี 16 หลัก")
            
        # เพื่อประโยชน์ในการทำ Routing และแสดงผล มักจะเก็บ 6 หลักแรก (BIN) และ 4 หลักท้ายไว้
        # เราจะ Tokenize แค่ 6 หลักตรงกลาง
        bin_part = cc_clean[:6]
        mid_part = cc_clean[6:12]
        last_part = cc_clean[12:]
        
        cipher = pyffx.String(self.secret_key, alphabet=self.alphabet, length=6)
        token_mid = cipher.encrypt(mid_part)
        
        return f"{bin_part}{token_mid}{last_part}"


# ==========================================
# ทดสอบการใช้งานจริง
# ==========================================
if __name__ == "__main__":
    # ⚠️ ในระบบจริง กุญแจนี้ต้องดึงมาจาก KMS (Key Management Service) หรือ Environment Variable 
    # ห้าม Hardcode ไว้ใน Source code เด็ดขาด
    MASTER_KEY = b"my-super-secret-key-256-bits!!" 
    
    tokenizer = DIYTokenizer(secret_key=MASTER_KEY)
    
    print("--- ทดสอบบัตรประชาชน ---")
    real_id = "1100200300400"
    token_id = tokenizer.tokenize_thai_id(real_id)
    recovered_id = tokenizer.detokenize_thai_id(token_id)
    
    print(f"ข้อมูลจริง: {real_id}")
    print(f"Token:    {token_id}")
    print(f"ถอดรหัส:   {recovered_id}")
    assert real_id == recovered_id
    
    print("\n--- ทดสอบบัตรเครดิต (Masking FPE) ---")
    real_cc = "4111222233334444"
    token_cc = tokenizer.tokenize_credit_card(real_cc)
    
    print(f"ข้อมูลจริง: {real_cc}")
    print(f"Token:    {token_cc} (เก็บ 6 หน้า 4 หลัง)")

--- ทดสอบบัตรประชาชน ---
ข้อมูลจริง: 1100200300400
Token:    8667307847498
ถอดรหัส:   1100200300400

--- ทดสอบบัตรเครดิต (Masking FPE) ---
ข้อมูลจริง: 4111222233334444
Token:    4111221051364444 (เก็บ 6 หน้า 4 หลัง)


In [4]:
from ff3 import FF3Cipher

# ต้องใช้ Key ขนาด 128-bit (16 bytes) หรือ 192-bit หรือ 256-bit เป็น Hex string
key = "2B7E151628AED2A6ABF7158809CF4F3C"
tweak = "D8E7920AFA330A73" # ค่า Tweak ช่วยเพิ่มความปลอดภัยแบบสุ่ม

cipher = FF3Cipher(key, tweak)

# เข้ารหัสบัตรประชาชน 13 หลัก
real_id = "1100200300400"
token = cipher.encrypt(real_id)

print(f"ข้อมูลต้นฉบับ: {real_id}")
print(f"Token (เข้ารหัส): {token}")

# ถอดรหัส
decrypted = cipher.decrypt(token)

# พิมพ์ผลลัพธ์การถอดรหัส
print(f"ข้อมูลถอดรหัส: {decrypted}")

ข้อมูลต้นฉบับ: 1100200300400
Token (เข้ารหัส): 8669010845197
ข้อมูลถอดรหัส: 1100200300400


HMAC (Hash-based Message Authentication Code) one_way_tokenize

In [ ]:
import hashlib
import hmac

def one_way_tokenize(data: str, secret_key: bytes) -> str:
    # ใช้ SHA-256 ร่วมกับ Secret Key (ป้องกันการถูกโจมตีแบบ Rainbow Table)
    hashed = hmac.new(secret_key, data.encode('utf-8'), hashlib.sha256)
    # ตัดมาใช้แค่ 16 ตัวอักษร เพื่อให้ Token ไม่ยาวเกินไป
    return f"tok_{hashed.hexdigest()[:100]}"

secret = b"my-analytics-salt-key"
print(one_way_tokenize("somchai@example.com", secret))

tok_bc492f06ec9e314dceaa0b8314e1b2bfea68be73ab95485bd8d8c784f731a558
